In [1]:
from ultralytics import YOLO
import numpy as np
import os
import torch

### 1.nii.gz -> png

In [3]:
import nibabel as nib
import numpy as np
from PIL import Image
import os

# 加载 .nii.gz 文件
nii_path = "/root/autodl-tmp/test/cbctZ轴追踪/image.nii.gz"  # 替换成你的路径
output_dir = "/root/autodl-tmp/test/cbctZ轴追踪/slices"          # 输出图像文件夹

# 创建输出目录
os.makedirs(output_dir, exist_ok=True)

# 读取nii.gz
img = nib.load(nii_path)
data = img.get_fdata()  # 获取numpy数组，shape通常为 (H, W, D)

# 归一化到0-255并转换为uint8
def normalize_slice(slice_2d):
    slice_2d = slice_2d - np.min(slice_2d)
    slice_2d = slice_2d / (np.max(slice_2d) + 1e-8)  # 避免除以0
    return (slice_2d * 255).astype(np.uint8)

# 遍历所有切片并保存
for i in range(data.shape[2]):  # 通常Z轴是最后一维
    slice_2d = data[:, :, i]
    slice_norm = normalize_slice(slice_2d)
    im = Image.fromarray(slice_norm)
    im.save(os.path.join(output_dir, f"slice_{i:03d}.png"))

print(f"保存完成，共保存 {data.shape[2]} 张图片到 {output_dir}/")


保存完成，共保存 401 张图片到 /root/autodl-tmp/test/cbctZ轴追踪/slices/


In [3]:
# Function to compute IOU
def compute_iou(box1, box2):
    """
    Compute the Intersection over Union (IOU) of two boxes.
    :param box1: [x_min, y_min, x_max, y_max]
    :param box2: [x_min, y_min, x_max, y_max]
    :return: IOU value
    """
    # Intersection coordinates
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2])
    inter_y2 = min(box1[3], box2[3])
    
    # Intersection area
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    
    # Box areas
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    # Union area
    union_area = area1 + area2 - inter_area
    
    # Avoid division by zero
    if union_area == 0:
        return 0.0
    
    # IOU calculation
    return inter_area / union_area
# Load a model
model = YOLO("/root/autodl-tmp/train_file/runs/segment/cbctZ轴追踪/weights/best.pt")  # load a custom model

data_path = '/root/autodl-tmp/test/cbctZ轴追踪/18757'

for file in os.listdir(data_path):
    if '18716' not in file:
        image = os.path.join(data_path, file)
        results = model(image)
        filename = file.split('.')[0]
        
        for result in results:
            # Step 1: Extract boxes and masks data
            boxes = result.boxes.data.cpu().numpy()  # Convert boxes to numpy array
            masks = result.masks.data.cpu().numpy() if result.masks else None  # Convert masks to numpy array if available
            
            # Step 2: Filter boxes based on IOU
            xyxy = boxes[:, :4]
            confs = boxes[:, -2]
            cls = boxes[:, -1]
            
            # IOU filtering
            keep_indices = set(range(len(boxes)))
            for i in range(len(boxes)):
                if i not in keep_indices:
                    continue
                for j in range(i + 1, len(boxes)):
                    if j not in keep_indices:
                        continue
                    iou = compute_iou(xyxy[i], xyxy[j])
                    if iou > 0.3:  # IOU threshold
                        if confs[i] >= confs[j]:
                            keep_indices.discard(j)
                        else:
                            keep_indices.discard(i)
                            break
            
            # Keep only selected indices after IOU filtering
            keep_indices = list(keep_indices)
            filtered_boxes = boxes[keep_indices]
            filtered_masks = masks[keep_indices] if masks is not None else None
            
            # Step 3: Further filter masks based on sum
            if filtered_masks is not None:
                mask_sums = filtered_masks.sum(axis=(1, 2))  # Compute the sum of each mask
                keep_indices = [i for i, m_sum in enumerate(mask_sums) if m_sum > 100]
                
                # Final filtering for masks and boxes
                filtered_boxes = filtered_boxes[keep_indices]
                filtered_masks = filtered_masks[keep_indices]
            
            # Step 4: Update result with filtered data
            result.boxes.data = torch.tensor(filtered_boxes).to(result.boxes.data.device)
            if filtered_masks is not None:
                result.masks.data = torch.tensor(filtered_masks).to(result.masks.data.device)
            
            # Save the filtered result image
            result.save(f'/root/autodl-tmp/test/cbctZ轴追踪/pred/{filename}_pred.jpg', labels=False, probs=False,boxes=False)





image 1/1 /root/autodl-tmp/test/cbctZ轴追踪/18757/18757_191.jpg: 640x640 12 teeths, 8.4ms
Speed: 2.2ms preprocess, 8.4ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /root/autodl-tmp/test/cbctZ轴追踪/18757/18757_192.jpg: 640x640 13 teeths, 8.8ms
Speed: 2.1ms preprocess, 8.8ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /root/autodl-tmp/test/cbctZ轴追踪/18757/18757_0.jpg: 640x640 (no detections), 11.4ms
Speed: 3.6ms preprocess, 11.4ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /root/autodl-tmp/test/cbctZ轴追踪/18757/18757_193.jpg: 640x640 13 teeths, 9.1ms
Speed: 2.2ms preprocess, 9.1ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /root/autodl-tmp/test/cbctZ轴追踪/18757/18757_1.jpg: 640x640 (no detections), 9.2ms
Speed: 2.0ms preprocess, 9.2ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /root/autodl-tmp/test/cbctZ轴追踪/18757/18757_194.jpg: 640x640

In [4]:
# 删除/root/autodl-tmp/test/cbctZ轴追踪下所有文件
import os

# 指定目录路径
directory = '/root/autodl-tmp/test/cbctZ轴追踪/pred'

# 检查目录是否存在
if os.path.exists(directory):
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)  # 删除文件或符号链接
            elif os.path.isdir(file_path):
                os.rmdir(file_path)  # 删除空目录
        except Exception as e:
            print(f"无法删除 {file_path}，原因: {e}")
    print(f"已删除 {directory} 下的所有文件。")
else:
    print(f"目录 {directory} 不存在。")


已删除 /root/autodl-tmp/test/cbctZ轴追踪/pred 下的所有文件。


In [ ]:
# 可视化box的中心点
import cv2
import numpy as np
import torch
from ultralytics import YOLO

# Function to compute IOU
def compute_iou(box1, box2):
    """
    Compute the Intersection over Union (IOU) of two boxes.
    :param box1: [x_min, y_min, x_max, y_max]
    :param box2: [x_min, y_min, x_max, y_max]
    :return: IOU value
    """
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2])
    inter_y2 = min(box1[3], box2[3])
    
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = area1 + area2 - inter_area
    
    if union_area == 0:
        return 0.0
    return inter_area / union_area

# Load a model
model = YOLO("/root/autodl-tmp/train_file/runs/segment/cbctZ轴追踪/weights/best.pt")

data_path = '/root/autodl-tmp/datasets/cbctZ轴跟踪/images/train/'
output_path = '/root/autodl-tmp/test/cbctZ轴追踪/'

for file in os.listdir(data_path):
    if '18716' in file:
        image_path = os.path.join(data_path, file)
        image = cv2.imread(image_path)  # Load original image
        results = model(image_path)
        filename = file.split('.')[0]
        
        for result in results:
            # Step 1: Extract boxes data
            boxes = result.boxes.data.cpu().numpy()  # Convert boxes to numpy array
            
            # Step 2: Filter boxes based on IOU
            xyxy = boxes[:, :4]
            confs = boxes[:, -2]
            cls = boxes[:, -1]
            
            keep_indices = set(range(len(boxes)))
            for i in range(len(boxes)):
                if i not in keep_indices:
                    continue
                for j in range(i + 1, len(boxes)):
                    if j not in keep_indices:
                        continue
                    iou = compute_iou(xyxy[i], xyxy[j])
                    if iou > 0.3:
                        if confs[i] >= confs[j]:
                            keep_indices.discard(j)
                        else:
                            keep_indices.discard(i)
                            break
            
            keep_indices = list(keep_indices)
            filtered_boxes = boxes[keep_indices]
            
            # Step 3: Draw box centers on the image
            for i, box in enumerate(filtered_boxes):
                # Compute box center
                x_min, y_min, x_max, y_max = box[:4]
                center_x = int((x_min + x_max) / 2)
                center_y = int((y_min + y_max) / 2)
                
                # Draw the center on the original image
                cv2.circle(image, (center_x, center_y), radius=5, color=(0, 255, 0), thickness=-1)  # Green dot
                cv2.putText(
                    image, f"Box {i}", (center_x + 10, center_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1
                )
        
        # Step 4: Save the image with box centers
        cv2.imwrite(os.path.join(output_path, f"{filename}_centered.jpg"), image)


In [6]:
import os
import numpy as np
import torch
from ultralytics import YOLO
import re

# Function to compute IOU
def compute_iou(box1, box2):
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2])
    inter_y2 = min(box1[3], box2[3])
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = area1 + area2 - inter_area
    return inter_area / union_area if union_area > 0 else 0.0

# Function to extract numeric part from filename
def extract_numeric_key(filename):
    match = re.search(r'_(\d+)\.', filename)
    return int(match.group(1)) if match else 0

# Initialize YOLO model
model = YOLO("/root/autodl-tmp/train_file/runs/segment/cbctZ轴追踪/weights/best.pt")
data_path = '/root/autodl-tmp/test/cbctZ轴追踪/slices'

# Initialize tracking list, finalized list, and ID counter
tracked_objects = []
finalized_tracks = []
current_id = 0

# Sort files based on numeric part of the filename
files = sorted(os.listdir(data_path), key=lambda x: extract_numeric_key(x))

for file_index, file in enumerate(files):
    if file.endswith(('.jpg', '.png', '.jpeg')):
        image_path = os.path.join(data_path, file)
        results = model(image_path)

        updated_ids = set()  # Track which IDs are updated in this iteration

        for result in results:
            # Step 1: Extract boxes and masks data
            boxes = result.boxes.data.cpu().numpy()
            masks = result.masks.data.cpu().numpy() if result.masks else None

            # Step 2: Filter boxes based on IOU
            xyxy = boxes[:, :4]
            confs = boxes[:, -2]
            cls = boxes[:, -1]

            keep_indices = set(range(len(boxes)))
            for i in range(len(boxes)):
                if i not in keep_indices:
                    continue
                for j in range(i + 1, len(boxes)):
                    if j not in keep_indices:
                        continue
                    iou = compute_iou(xyxy[i], xyxy[j])
                    if iou > 0.3:
                        if confs[i] >= confs[j]:
                            keep_indices.discard(j)
                        else:
                            keep_indices.discard(i)
                            break

            keep_indices = list(keep_indices)
            filtered_boxes = boxes[keep_indices]
            filtered_masks = masks[keep_indices] if masks is not None else None

            if filtered_masks is not None:
                mask_sums = filtered_masks.sum(axis=(1, 2))
                keep_indices = [i for i, m_sum in enumerate(mask_sums) if m_sum > 100]

                filtered_boxes = filtered_boxes[keep_indices]
                filtered_masks = filtered_masks[keep_indices]

                # Step 3: Match and update tracked_objects
                for i, (mask, box) in enumerate(zip(filtered_masks, filtered_boxes)):
                    matched = False
                    for tracked in tracked_objects:
                        tracked_id, tracked_masks, tracked_xyxys, tracked_z = tracked
                        iou = compute_iou(box[:4], tracked_xyxys[-1])

                        if iou > 0.7:  # Match existing tracked object
                            tracked_masks.append(mask)
                            tracked_xyxys.append(box[:4])
                            tracked_z.append(file_index)
                            updated_ids.add(tracked_id)
                            matched = True
                            break

                    if not matched:
                        # Add new tracked object
                        tracked_objects.append([current_id, [mask], [box[:4]], [file_index]])
                        updated_ids.add(current_id)
                        current_id += 1

        # Step 4: Finalize unupdated tracked objects
        for tracked in tracked_objects[:]:  # Iterate over a copy to allow removal
            tracked_id = tracked[0]
            if tracked_id not in updated_ids:
                finalized_tracks.append({
                    tracked_id: {
                        "masks": tracked[1],
                        "boxes": tracked[2],
                        "z": tracked[3]
                    }
                })
                tracked_objects.remove(tracked)
# Filter finalized_tracks to remove entries where the length of `z` is less than or equal to 3
finalized_tracks = [
    track for track in finalized_tracks
    if list(track.values())[0]["z"] and len(list(track.values())[0]["z"]) > 10
]

# Output the filtered finalized tracks
print("Filtered Finalized Tracks (Z length > 3):")
print(len(finalized_tracks))


image 1/1 /root/autodl-tmp/test/cbctZ轴追踪/slices/slice_000.png: 640x640 (no detections), 7.4ms
Speed: 7.2ms preprocess, 7.4ms inference, 73.9ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /root/autodl-tmp/test/cbctZ轴追踪/slices/slice_001.png: 640x640 (no detections), 7.7ms
Speed: 2.7ms preprocess, 7.7ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /root/autodl-tmp/test/cbctZ轴追踪/slices/slice_002.png: 640x640 (no detections), 6.3ms
Speed: 1.5ms preprocess, 6.3ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /root/autodl-tmp/test/cbctZ轴追踪/slices/slice_003.png: 640x640 (no detections), 7.3ms
Speed: 1.8ms preprocess, 7.3ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /root/autodl-tmp/test/cbctZ轴追踪/slices/slice_004.png: 640x640 (no detections), 6.8ms
Speed: 1.6ms preprocess, 6.8ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /root/autodl-tmp/test/cbctZ轴追踪/sl

In [8]:
import nibabel as nib
import numpy as np

# Load the original NIfTI file
file_path = "/root/autodl-tmp/test/cbctZ轴追踪/image.nii.gz"
nii = nib.load(file_path)
original_shape = nii.get_fdata().shape
print(original_shape)
# Initialize a new NIfTI data array with float64 type
new_data = np.zeros(original_shape, dtype=np.float64)

from skimage.transform import resize

# Function to resize mask to match target shape
def resize_mask(mask, target_shape):
    resized_mask = resize(mask, target_shape, mode='constant', preserve_range=True, anti_aliasing=True)
    return resized_mask

# Populate the new NIfTI array with finalized_tracks data
for track in finalized_tracks:
    for track_id, track_data in track.items():
        masks = track_data["masks"]  # List of masks
        z_indices = track_data["z"]  # Corresponding z indices

        for mask, z in zip(masks, z_indices):
            # Ensure z is within bounds
            if 0 <= z < original_shape[2]:
                # Resize mask to match NIfTI dimensions
                resized_mask = resize_mask(mask, (original_shape[0], original_shape[1]))
                resized_mask = resized_mask > 0  # Convert to boolean mask

                # Add mask to the corresponding z slice
                new_data[:, :, z][resized_mask] = float(track_id)

# Save the new NIfTI file
output_file = "/root/autodl-tmp/test/cbctZ轴追踪/pred.nii.gz"
new_nii = nib.Nifti1Image(new_data, affine=nii.affine)
nib.save(new_nii, output_file)

print(f"Saved new NIfTI file with float64 format: {output_file}")


(559, 559, 401)
Saved new NIfTI file with float64 format: /root/autodl-tmp/test/cbctZ轴追踪/pred.nii.gz
